# Week 6: Data Cleaning & Normalization — PHASE 4: Processing Stages

*Core Mastery: "I can clean messy real-world data by handling missing values, outliers, and inconsistencies"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Explain why data cleaning is a critical stage in every data pipeline
2. Detect missing values in various disguises: `None`, `""`, `"N/A"`, `"nan"`, `"?"`
3. Handle missing values by dropping rows, filling with mean, or forward-filling
4. Detect statistical outliers using the **IQR method** and **z-score** concept
5. Handle outliers by clipping, removing, or flagging them
6. Implement a `safe_float()` function for robust type conversion
7. Apply **min-max normalization** and **z-score standardization** to numeric data
8. Build a complete cleaning pipeline: read → validate → clean → normalize → export
9. Quantify data quality with completeness and validity metrics
10. Apply cleaning techniques to messy sensor and student data

## 🎯 Core Mastery Connection

Last week you learned to read and write structured data in CSV and JSON formats, and to validate data against a schema. But validation only **detects** problems — it does not **fix** them. Real-world data from sensors, surveys, and databases is almost always messy: readings are missing, values spike due to sensor glitches, and strings appear where numbers should be.

This week you add the **cleaning and normalization** stage to your pipeline. After reading and validating, you will systematically handle missing values, detect and manage outliers, ensure type consistency, and normalize scales. These skills transform raw, unreliable data into analysis-ready datasets.

---
## Part 1: Why Clean? — The Reality of Messy Data

Real data is messy. Here are the most common problems you will encounter:

| Problem | Example | Impact |
|---------|---------|--------|
| Missing values | `""`, `None`, `"N/A"` | Crashes calculations, biased results |
| Wrong types | `"abc"` in a numeric column | `ValueError` on conversion |
| Outliers | Temperature = 999.9 (sensor glitch) | Skews averages and statistics |
| Inconsistent formats | `"22.5"`, `"22,5"`, `"22.5 C"` | Parse failures |
| Duplicates | Same reading recorded twice | Inflated counts |
| Inconsistent categories | `"OK"`, `"ok"`, `"Ok"`, `"O.K."` | Splits one category into many |

**The cleaning pipeline sits between reading and analysis:**

```
Read → Validate → CLEAN → Normalize → Analyze → Write
```

Without cleaning, your analysis results will be wrong — often silently wrong, which is worse than a crash.

---
## Part 2: Detecting Missing Values

Missing values come in many disguises. A robust detection function must check for all of them.

| Representation | Type | Example |
|----------------|------|---------|
| `None` | NoneType | Python's null |
| `""` | str | Empty string |
| `"N/A"` | str | Common placeholder |
| `"nan"` | str | Not a Number |
| `"?"` | str | Survey placeholder |
| `"-"` | str | Dash as missing |
| `"null"` | str | JSON null as string |

**Figure 2.1** — `is_missing()` function

In [ ]:
MISSING_MARKERS = {"", "n/a", "na", "nan", "null", "none", "?", "-", "--", "missing"}

def is_missing(value):
    """Check if a value represents missing data."""
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in MISSING_MARKERS:
        return True
    return False

# Test with various inputs
test_values = [None, "", "N/A", "nan", "?", "-", "23.5", "OK", "null", 0, "  na  "]
for v in test_values:
    print(f"  is_missing({str(v):>8}) = {is_missing(v)}")

**Figure 2.2** — Scanning a dataset for missing values

In [ ]:
data = [
    {"sensor": "S01", "temp": "22.5", "humidity": "55",  "status": "OK"},
    {"sensor": "S02", "temp": "N/A",  "humidity": "42",  "status": "WARN"},
    {"sensor": "S03", "temp": "19.8", "humidity": "",    "status": "OK"},
    {"sensor": "S04", "temp": "?",    "humidity": "38",  "status": None},
    {"sensor": "S05", "temp": "23.1", "humidity": "nan", "status": "OK"},
]

print("Missing Value Report")
print("-" * 45)
for i, row in enumerate(data):
    missing_fields = [k for k, v in row.items() if is_missing(v)]
    if missing_fields:
        print(f"  Row {i} ({row['sensor']}): missing {missing_fields}")

# Column-level summary
fields = ["sensor", "temp", "humidity", "status"]
print("\nColumn Completeness:")
for field in fields:
    present = sum(1 for row in data if not is_missing(row.get(field)))
    pct = present / len(data) * 100
    print(f"  {field:<10} {present}/{len(data)} ({pct:.0f}%)")

**Figure 2.3** — Counting missing values per column

In [ ]:
import random
random.seed(42)

# Generate messy dataset
sensors = []
for i in range(20):
    row = {"id": f"S{i+1:02d}"}
    row["temp"] = round(random.gauss(25, 5), 1) if random.random() > 0.15 else random.choice(["N/A", "", None, "?"])
    row["humidity"] = round(random.uniform(30, 70), 1) if random.random() > 0.1 else "nan"
    row["pressure"] = round(random.gauss(1013, 5), 1) if random.random() > 0.2 else "-"
    sensors.append(row)

# Count missing per column
columns = ["temp", "humidity", "pressure"]
print(f"Dataset: {len(sensors)} rows")
print("-" * 40)
for col in columns:
    n_missing = sum(1 for row in sensors if is_missing(row[col]))
    n_present = len(sensors) - n_missing
    print(f"  {col:<12} present={n_present:>2}  missing={n_missing:>2}  ({n_missing/len(sensors)*100:.0f}% missing)")

---
## Part 3: Handling Missing Values

Three common strategies for handling missing values:

| Strategy | When to Use | Pros | Cons |
|----------|-------------|------|------|
| **Drop row** | Few missing, enough data | Simple, no bias | Loses data |
| **Fill with mean** | Numeric column, random missingness | Preserves dataset size | Reduces variance |
| **Forward fill** | Time series, sequential data | Preserves trends | May propagate stale values |

**Figure 3.1** — Drop rows with missing values

In [ ]:
data = [
    {"sensor": "S01", "temp": "22.5", "status": "OK"},
    {"sensor": "S02", "temp": "N/A",  "status": "WARN"},
    {"sensor": "S03", "temp": "19.8", "status": "OK"},
    {"sensor": "S04", "temp": "?",    "status": None},
    {"sensor": "S05", "temp": "23.1", "status": "OK"},
]

def drop_rows_with_missing(data, columns):
    """Remove rows that have missing values in any of the specified columns."""
    clean = []
    dropped = 0
    for row in data:
        if any(is_missing(row.get(col)) for col in columns):
            dropped += 1
        else:
            clean.append(row)
    return clean, dropped

clean, dropped = drop_rows_with_missing(data, ["temp", "status"])
print(f"Before: {len(data)} rows")
print(f"Dropped: {dropped}")
print(f"After: {len(clean)} rows")
for row in clean:
    print(f"  {row}")

**Figure 3.2** — Fill missing values with column mean

In [ ]:
def fill_with_mean(data, column):
    """Replace missing values in a numeric column with the column mean."""
    # Collect non-missing numeric values
    values = []
    for row in data:
        if not is_missing(row.get(column)):
            try:
                values.append(float(row[column]))
            except (ValueError, TypeError):
                pass

    if not values:
        return data, 0

    mean_val = sum(values) / len(values)
    filled = 0
    for row in data:
        if is_missing(row.get(column)):
            row[column] = round(mean_val, 2)
            filled += 1
        else:
            try:
                row[column] = float(row[column])
            except (ValueError, TypeError):
                row[column] = round(mean_val, 2)
                filled += 1
    return data, filled

data = [
    {"id": "S01", "temp": "22.5"},
    {"id": "S02", "temp": "N/A"},
    {"id": "S03", "temp": "19.8"},
    {"id": "S04", "temp": ""},
    {"id": "S05", "temp": "23.1"},
    {"id": "S06", "temp": "21.0"},
]

print("Before:", [(r["id"], r["temp"]) for r in data])
data, n = fill_with_mean(data, "temp")
print(f"Filled {n} missing values with mean")
print("After: ", [(r["id"], r["temp"]) for r in data])

**Figure 3.3** — Forward fill for time-series data

In [ ]:
def forward_fill(data, column):
    """Fill missing values with the last known value (forward fill)."""
    last_value = None
    filled = 0
    for row in data:
        if is_missing(row.get(column)):
            if last_value is not None:
                row[column] = last_value
                filled += 1
        else:
            last_value = row[column]
    return data, filled

time_series = [
    {"time": "10:00", "temp": "22.5"},
    {"time": "10:05", "temp": "22.8"},
    {"time": "10:10", "temp": "N/A"},
    {"time": "10:15", "temp": "N/A"},
    {"time": "10:20", "temp": "23.5"},
    {"time": "10:25", "temp": ""},
    {"time": "10:30", "temp": "24.0"},
]

print("Before:", [(r["time"], r["temp"]) for r in time_series])
time_series, n = forward_fill(time_series, "temp")
print(f"Forward-filled {n} values")
print("After: ", [(r["time"], r["temp"]) for r in time_series])

---
## Part 4: Detecting Outliers — IQR and Z-Score

An **outlier** is a data point that is significantly different from the rest. Two common detection methods:

**IQR Method (Interquartile Range):**
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1
- Lower bound = Q1 - 1.5 * IQR
- Upper bound = Q3 + 1.5 * IQR
- Any value outside [lower, upper] is an outlier

**Z-Score Method:**
- z = (x - mean) / std_dev
- |z| > 2 or |z| > 3 is typically considered an outlier

The IQR method is more **robust** because it uses medians, not means, making it resistant to extreme values.

**Figure 4.1** — Implementing IQR outlier detection

In [ ]:
def get_quartiles(values):
    """Compute Q1, Q2 (median), Q3 for a sorted list."""
    s = sorted(values)
    n = len(s)
    q2 = s[n // 2] if n % 2 != 0 else (s[n//2 - 1] + s[n//2]) / 2
    lower = s[:n // 2]
    upper = s[n // 2 + (n % 2):]
    q1 = lower[len(lower)//2] if len(lower) % 2 != 0 else (lower[len(lower)//2-1] + lower[len(lower)//2]) / 2
    q3 = upper[len(upper)//2] if len(upper) % 2 != 0 else (upper[len(upper)//2-1] + upper[len(upper)//2]) / 2
    return q1, q2, q3

def detect_outliers_iqr(values, factor=1.5):
    """Return indices of outliers using IQR method."""
    q1, q2, q3 = get_quartiles(values)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    outliers = [(i, v) for i, v in enumerate(values) if v < lower or v > upper]
    return outliers, lower, upper, q1, q3, iqr

# Test with sensor data (one spike outlier)
temps = [22.1, 22.5, 23.0, 22.8, 23.2, 21.9, 22.7, 99.5, 22.4, 23.1, 22.0, 22.6]
outliers, lo, hi, q1, q3, iqr = detect_outliers_iqr(temps)

print(f"Data: {temps}")
print(f"Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}")
print(f"Bounds: [{lo:.1f}, {hi:.1f}]")
print(f"Outliers: {outliers}")

**Figure 4.2** — Z-score outlier detection

In [ ]:
import math

def mean(values):
    return sum(values) / len(values)

def std_dev(values):
    m = mean(values)
    variance = sum((x - m) ** 2 for x in values) / len(values)
    return math.sqrt(variance)

def detect_outliers_zscore(values, threshold=2.0):
    """Return indices of outliers using z-score method."""
    m = mean(values)
    sd = std_dev(values)
    if sd == 0:
        return []
    outliers = []
    for i, v in enumerate(values):
        z = (v - m) / sd
        if abs(z) > threshold:
            outliers.append((i, v, round(z, 2)))
    return outliers

temps = [22.1, 22.5, 23.0, 22.8, 23.2, 21.9, 22.7, 99.5, 22.4, 23.1, 22.0, 22.6]
outliers = detect_outliers_zscore(temps, threshold=2.0)

print(f"Mean: {mean(temps):.1f}, Std: {std_dev(temps):.1f}")
print(f"Outliers (|z| > 2.0):")
for idx, val, z in outliers:
    print(f"  Index {idx}: value={val}, z-score={z}")

**Figure 4.3** — Comparing IQR and z-score on real-ish data

In [ ]:
import random
random.seed(100)

# Generate data with a few outliers injected
normal_data = [round(random.gauss(50, 5), 1) for _ in range(30)]
normal_data[5] = 120.0   # Outlier
normal_data[15] = -10.0  # Outlier
normal_data[25] = 95.0   # Borderline

print("Dataset (30 values, 3 anomalies injected at indices 5, 15, 25)")
print(f"  Range: [{min(normal_data):.1f}, {max(normal_data):.1f}]")

iqr_outliers, lo, hi, _, _, _ = detect_outliers_iqr(normal_data)
z_outliers = detect_outliers_zscore(normal_data, threshold=2.0)

print(f"\nIQR Method: bounds=[{lo:.1f}, {hi:.1f}]")
print(f"  Found {len(iqr_outliers)} outliers: {[(i,v) for i,v in iqr_outliers]}")

print(f"\nZ-Score Method (|z|>2):")
print(f"  Found {len(z_outliers)} outliers: {[(i,v) for i,v,z in z_outliers]}")

---
## Part 5: Handling Outliers — Clip, Remove, or Flag

Once detected, outliers can be handled in three ways:

| Strategy | Description | When to Use |
|----------|-------------|-------------|
| **Clip** | Replace with boundary value | Preserve dataset size, known limits |
| **Remove** | Drop the row entirely | Clearly erroneous data |
| **Flag** | Add a column marking outliers | Want to keep data but track issues |

**Figure 5.1** — Clipping outliers to bounds

In [ ]:
def clip_outliers(values, lower=None, upper=None):
    """Clip values to [lower, upper] range."""
    if lower is None or upper is None:
        q1, _, q3 = get_quartiles(values)
        iqr = q3 - q1
        if lower is None: lower = q1 - 1.5 * iqr
        if upper is None: upper = q3 + 1.5 * iqr

    clipped = []
    n_clipped = 0
    for v in values:
        if v < lower:
            clipped.append(lower)
            n_clipped += 1
        elif v > upper:
            clipped.append(upper)
            n_clipped += 1
        else:
            clipped.append(v)
    return clipped, n_clipped, lower, upper

temps = [22.1, 22.5, 23.0, 22.8, 99.5, 21.9, 22.7, -5.0, 22.4, 23.1]
clipped, n, lo, hi = clip_outliers(temps)

print(f"Original: {temps}")
print(f"Clipped:  {[round(v,1) for v in clipped]}")
print(f"Bounds: [{lo:.1f}, {hi:.1f}], Clipped {n} values")

**Figure 5.2** — Removing outlier rows from a dataset

In [ ]:
def remove_outlier_rows(data, column):
    """Remove rows where the specified column value is an IQR outlier."""
    values = [float(row[column]) for row in data]
    outlier_indices = set(i for i, _ in detect_outliers_iqr(values)[0])

    clean = [row for i, row in enumerate(data) if i not in outlier_indices]
    return clean, len(outlier_indices)

sensor_data = [
    {"id": "S01", "temp": 22.5}, {"id": "S02", "temp": 23.1},
    {"id": "S03", "temp": 99.5}, {"id": "S04", "temp": 22.8},
    {"id": "S05", "temp": 21.9}, {"id": "S06", "temp": -10.0},
    {"id": "S07", "temp": 22.4}, {"id": "S08", "temp": 23.0},
]

clean, n_removed = remove_outlier_rows(sensor_data, "temp")
print(f"Before: {len(sensor_data)} rows")
print(f"Removed: {n_removed} outlier rows")
print(f"After: {len(clean)} rows")
for row in clean:
    print(f"  {row['id']}: {row['temp']}")

**Figure 5.3** — Flagging outliers with a separate column

In [ ]:
def flag_outliers(data, column, flag_col="is_outlier"):
    """Add a boolean flag column indicating outliers."""
    values = [float(row[column]) for row in data]
    outlier_indices = set(i for i, _ in detect_outliers_iqr(values)[0])

    for i, row in enumerate(data):
        row[flag_col] = i in outlier_indices
    return data

readings = [
    {"id": "S01", "temp": 22.5}, {"id": "S02", "temp": 23.1},
    {"id": "S03", "temp": 99.5}, {"id": "S04", "temp": 22.8},
    {"id": "S05", "temp": 21.9}, {"id": "S06", "temp": 22.4},
]

flag_outliers(readings, "temp")
print("Flagged Data:")
for row in readings:
    marker = " ⚠️ OUTLIER" if row["is_outlier"] else ""
    print(f"  {row['id']}: {row['temp']:>6.1f}{marker}")

---
## Part 6: Type Consistency — Safe Conversions

Real data often has strings where numbers should be, or mixed formats. A `safe_float()` function handles all edge cases gracefully.

**Figure 6.1** — `safe_float()` function

In [ ]:
def safe_float(value, default=None):
    """Safely convert a value to float. Returns default if conversion fails."""
    if value is None:
        return default
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        cleaned = value.strip().replace(",", ".")  # Handle Turkish decimal comma
        cleaned = cleaned.rstrip(" CcFf°")  # Remove unit suffixes
        if cleaned.lower() in ("", "n/a", "na", "nan", "null", "none", "?", "-"):
            return default
        try:
            return float(cleaned)
        except ValueError:
            return default
    return default

# Test various inputs
test_cases = ["23.5", "22,5", " 19.8 ", "N/A", "", None, "25.0 C", "abc", 42, "nan", "?"]
for v in test_cases:
    result = safe_float(v, default=-999)
    print(f"  safe_float({str(v):>10}) = {result}")

**Figure 6.2** — Converting an entire column

In [ ]:
def convert_column(data, column, converter=safe_float, default=None):
    """Convert all values in a column using the given converter."""
    errors = 0
    for row in data:
        original = row[column]
        converted = converter(original, default)
        if converted == default and not is_missing(original):
            errors += 1
        row[column] = converted
    return data, errors

records = [
    {"id": "S01", "temp": "22.5",  "humidity": "55"},
    {"id": "S02", "temp": "abc",   "humidity": "42"},
    {"id": "S03", "temp": "19,8",  "humidity": "N/A"},
    {"id": "S04", "temp": "",      "humidity": "38"},
    {"id": "S05", "temp": "23.1C", "humidity": "61"},
]

print("Before conversion:")
for r in records:
    print(f"  {r['id']}: temp={r['temp']!r}, humidity={r['humidity']!r}")

convert_column(records, "temp", safe_float, None)
convert_column(records, "humidity", safe_float, None)

print("\nAfter conversion:")
for r in records:
    print(f"  {r['id']}: temp={r['temp']}, humidity={r['humidity']}")

**Figure 6.3** — Cleaning string categories

In [ ]:
def normalize_category(value, valid_categories, mapping=None):
    """Normalize a category string to a standard form."""
    if is_missing(value):
        return None
    cleaned = value.strip().upper()
    if mapping and cleaned in mapping:
        cleaned = mapping[cleaned]
    if cleaned in valid_categories:
        return cleaned
    return None

STATUS_VALID = {"OK", "WARN", "CRITICAL", "OFFLINE"}
STATUS_MAPPING = {
    "O.K.": "OK", "OKAY": "OK", "WARNING": "WARN",
    "ERROR": "CRITICAL", "ERR": "CRITICAL", "OFF": "OFFLINE"
}

test_statuses = ["OK", "ok", " Ok ", "O.K.", "WARNING", "ERROR", "err", "off", "banana", ""]
print("Status Normalization:")
for s in test_statuses:
    result = normalize_category(s, STATUS_VALID, STATUS_MAPPING)
    print(f"  {s!r:>12} -> {result}")

---
## Part 7: Normalization — Scaling to Standard Ranges

**Normalization** rescales numeric values to a standard range, making different measurements comparable.

**Min-Max Normalization** (scales to [0, 1]):

$$x_{norm} = \\frac{x - x_{min}}{x_{max} - x_{min}}$$

**Z-Score Standardization** (scales to mean=0, std=1):

$$z = \\frac{x - \\mu}{\\sigma}$$

| Method | Output Range | Use When |
|--------|-------------|----------|
| Min-Max | [0, 1] | Need bounded values, known min/max |
| Z-Score | unbounded | Comparing different distributions |

**Figure 7.1** — Min-max normalization

In [ ]:
def min_max_normalize(values):
    """Normalize values to [0, 1] range."""
    mn = min(values)
    mx = max(values)
    if mx == mn:
        return [0.5] * len(values)  # All same value
    return [round((v - mn) / (mx - mn), 4) for v in values]

temps = [18.2, 22.5, 25.1, 30.2, 27.6, 21.0, 24.4, 19.8]
normalized = min_max_normalize(temps)

print("Min-Max Normalization (Temperature)")
print(f"  Original range: [{min(temps)}, {max(temps)}]")
print(f"  Normalized range: [{min(normalized)}, {max(normalized)}]")
print()
for t, n in zip(temps, normalized):
    bar = "█" * int(n * 30)
    print(f"  {t:>5.1f} -> {n:.4f}  {bar}")

**Figure 7.2** — Z-score standardization

In [ ]:
import math

def z_score_normalize(values):
    """Standardize values to mean=0, std=1."""
    m = sum(values) / len(values)
    variance = sum((x - m) ** 2 for x in values) / len(values)
    sd = math.sqrt(variance)
    if sd == 0:
        return [0.0] * len(values)
    return [round((v - m) / sd, 4) for v in values]

temps = [18.2, 22.5, 25.1, 30.2, 27.6, 21.0, 24.4, 19.8]
z_scores = z_score_normalize(temps)

m = sum(temps) / len(temps)
print("Z-Score Standardization (Temperature)")
print(f"  Mean: {m:.1f}")
print()
for t, z in zip(temps, z_scores):
    direction = "+" if z >= 0 else ""
    print(f"  {t:>5.1f} -> z={direction}{z:.4f}")

**Figure 7.3** — Normalizing multiple columns

In [ ]:
sensor_data = [
    {"id": "S01", "temp": 22.5, "humidity": 55, "pressure": 1013},
    {"id": "S02", "temp": 25.1, "humidity": 42, "pressure": 1008},
    {"id": "S03", "temp": 19.8, "humidity": 68, "pressure": 1015},
    {"id": "S04", "temp": 28.3, "humidity": 38, "pressure": 1010},
    {"id": "S05", "temp": 21.0, "humidity": 61, "pressure": 1012},
]

columns = ["temp", "humidity", "pressure"]

for col in columns:
    values = [row[col] for row in sensor_data]
    normed = min_max_normalize(values)
    for row, n in zip(sensor_data, normed):
        row[f"{col}_norm"] = n

print("Normalized Sensor Data:")
print(f"  {'ID':<5} {'temp':>5} {'t_n':>5} {'hum':>5} {'h_n':>5} {'pres':>6} {'p_n':>5}")
print("  " + "-" * 42)
for row in sensor_data:
    print(f"  {row['id']:<5} {row['temp']:>5.1f} {row['temp_norm']:>5.3f} "
          f"{row['humidity']:>5.0f} {row['humidity_norm']:>5.3f} "
          f"{row['pressure']:>6.0f} {row['pressure_norm']:>5.3f}")

---
## Part 8: Building a Complete Cleaning Pipeline

Now we chain all cleaning functions into a single pipeline:

**Read → Validate → Clean Missing → Remove Outliers → Normalize → Export**

Each stage takes data in and passes cleaned data to the next stage.

**Figure 8.1** — Full cleaning pipeline for sensor data

In [ ]:
import csv, json, io

# ── RAW INPUT (messy!) ──
raw_csv = """sensor_id,temperature,humidity,status
S01,22.5,55,OK
S02,N/A,42,WARN
S03,19.8,,OK
S04,abc,38,CRITICAL
S05,99.5,61,OK
S06,23.1,nan,ok
S07,21.0,58,OK
S08,-50,45,OK
S09,22.8,52,OK
S10,24.3,49,"""

print("═══ CLEANING PIPELINE ═══")

# STEP 1: Read
reader = csv.DictReader(io.StringIO(raw_csv))
data = list(reader)
print(f"\n1. READ: {len(data)} rows loaded")

# STEP 2: Clean types
for row in data:
    row["temperature"] = safe_float(row["temperature"])
    row["humidity"] = safe_float(row["humidity"])
    row["status"] = normalize_category(row.get("status",""), {"OK","WARN","CRITICAL"}, {"WARNING":"WARN","ERROR":"CRITICAL"})

n_temp_none = sum(1 for r in data if r["temperature"] is None)
n_hum_none = sum(1 for r in data if r["humidity"] is None)
print(f"2. TYPE CLEAN: temp missing={n_temp_none}, humidity missing={n_hum_none}")

# STEP 3: Fill missing humidity with mean, drop rows with missing temp
data, filled = fill_with_mean(data, "humidity")
print(f"3. FILL: {filled} humidity values filled with mean")
data = [r for r in data if r["temperature"] is not None]
print(f"   DROPPED rows with missing temp -> {len(data)} rows remain")

# STEP 4: Remove outlier temperatures
temps_before = [r["temperature"] for r in data]
outliers_found, lo, hi, _, _, _ = detect_outliers_iqr(temps_before)
data = [r for i, r in enumerate(data) if i not in set(j for j, _ in outliers_found)]
print(f"4. OUTLIERS: bounds=[{lo:.1f}, {hi:.1f}], removed {len(outliers_found)} -> {len(data)} rows")

# STEP 5: Normalize temperature
temps = [r["temperature"] for r in data]
normed = min_max_normalize(temps)
for r, n in zip(data, normed):
    r["temp_normalized"] = n
print(f"5. NORMALIZE: temperature scaled to [0, 1]")

# STEP 6: Export
print(f"\n═══ FINAL OUTPUT ({len(data)} clean rows) ═══")
for r in data:
    print(f"  {r['sensor_id']}  temp={r['temperature']:>5.1f}  norm={r['temp_normalized']:.3f}  "
          f"hum={r['humidity']:>5.1f}  [{r['status'] or 'N/A'}]")

**Figure 8.2** — Pipeline with quality metrics

In [ ]:
import csv, io

raw_csv = """student_id,name,dept,midterm,final
101,Zeynep Kaya,EE,78,85
102,Ali Demir,ME,N/A,72
103,Fatma Celik,CS,92,95
104,,EE,70,999
105,Elif Yildiz,ME,88,91
106,Burak Ozkan,XX,55,60
107,Mehmet Kara,CS,abc,80
108,Ayse Turk,EE,82,"""

print("═══ STUDENT DATA CLEANING ═══\n")

reader = csv.DictReader(io.StringIO(raw_csv))
data = list(reader)
total_start = len(data)
total_fields = total_start * 5  # 5 columns

# Count issues
issues = {"missing": 0, "type_error": 0, "out_of_range": 0, "invalid_category": 0}

for row in data:
    row["midterm"] = safe_float(row["midterm"])
    row["final"] = safe_float(row["final"])
    if row["midterm"] is None: issues["missing"] += 1
    if row["final"] is None: issues["missing"] += 1
    if is_missing(row["name"]): issues["missing"] += 1

    # Range check
    for f in ["midterm", "final"]:
        if row[f] is not None and (row[f] < 0 or row[f] > 100):
            issues["out_of_range"] += 1
            row[f] = None  # Mark as invalid

    # Dept check
    if row["dept"] not in ("EE", "ME", "CS", "CE", "IE"):
        issues["invalid_category"] += 1

# Fill missing scores with mean
for col in ["midterm", "final"]:
    vals = [r[col] for r in data if r[col] is not None]
    if vals:
        mean_val = sum(vals) / len(vals)
        for r in data:
            if r[col] is None:
                r[col] = round(mean_val, 1)

# Drop invalid rows (missing name or invalid dept)
clean = [r for r in data if not is_missing(r["name"]) and r["dept"] in ("EE","ME","CS","CE","IE")]

# Compute averages
for r in clean:
    r["average"] = round(r["midterm"] * 0.4 + r["final"] * 0.6, 1)

print("Quality Report:")
for issue, count in issues.items():
    print(f"  {issue:<20} {count}")
print(f"\nRows: {total_start} -> {len(clean)} ({total_start-len(clean)} removed)")
print(f"\nClean Data:")
for r in clean:
    print(f"  {r['student_id']}  {r['name']:<16} {r['dept']}  "
          f"mid={r['midterm']:>5.1f}  fin={r['final']:>5.1f}  avg={r['average']:.1f}")

---
## Exercises

### Exercise 1: Detect Missing Values

Given `data = [{'id':'S01','temp':'22.5'},{'id':'S02','temp':'N/A'},{'id':'S03','temp':''},{'id':'S04','temp':None},{'id':'S05','temp':'23.1'}]`, write code that prints which sensors have missing temperature values using the `is_missing()` function.

**Expected output:** S02, S03, S04 should be flagged as missing.

<details><summary>💡 Hint</summary>

Loop through data and call `is_missing(row['temp'])` for each row.

</details>

In [ ]:
# ✏️ [EX1]


### Exercise 2: Column Completeness

Given messy data with 5 columns and 10 rows (create it with some N/A, None, empty values), calculate and print the completeness percentage for each column.

**Expected output:** Output like: temp 80%, humidity 70%, status 90% etc.

<details><summary>💡 Hint</summary>

Count non-missing values per column and divide by total rows.

</details>

In [ ]:
# ✏️ [EX2]


### Exercise 3: Drop Missing Rows

Given `data = [{'name':'Ahmet','score':85},{'name':'Ayse','score':None},{'name':'Burak','score':72},{'name':'','score':90},{'name':'Elif','score':88}]`, drop rows where name OR score is missing. Print before and after counts.

**Expected output:** Before: 5, After: 3 (Ahmet, Burak, Elif).

<details><summary>💡 Hint</summary>

Check `is_missing()` for both 'name' and 'score' fields.

</details>

In [ ]:
# ✏️ [EX3]


### Exercise 4: Fill with Mean

Given `temps = [22.5, None, 19.8, None, 23.1, 21.0, None, 22.8]`, replace None values with the mean of non-None values. Print the list before and after.

**Expected output:** Mean of [22.5, 19.8, 23.1, 21.0, 22.8] = 21.84. Fill Nones with 21.84.

<details><summary>💡 Hint</summary>

First compute mean of non-None values, then replace Nones.

</details>

In [ ]:
# ✏️ [EX4]


### Exercise 5: Forward Fill

Given `readings = [22.5, 22.8, None, None, 23.5, None, 24.0, None, None, 25.1]`, implement forward fill to replace each None with the previous non-None value. Print before and after.

**Expected output:** [22.5, 22.8, 22.8, 22.8, 23.5, 23.5, 24.0, 24.0, 24.0, 25.1]

<details><summary>💡 Hint</summary>

Track `last_value` and update it when you encounter a non-None value.

</details>

In [ ]:
# ✏️ [EX5]


### Exercise 6: IQR Outlier Detection

Given `data = [22, 23, 21, 24, 22, 23, 99, 22, 21, 23, 24, -15, 22, 23]`, use the IQR method to find outliers. Print Q1, Q3, IQR, bounds, and the outlier values.

**Expected output:** 99 and -15 should be detected as outliers.

<details><summary>💡 Hint</summary>

Sort the data, find Q1/Q3, compute IQR, and check bounds.

</details>

In [ ]:
# ✏️ [EX6]


### Exercise 7: Clip Outliers

Using the same data from Exercise 6, clip outlier values to the IQR bounds instead of removing them. Print the original and clipped lists side by side.

**Expected output:** 99 and -15 should be replaced with the upper and lower bounds.

<details><summary>💡 Hint</summary>

Use `min(max(v, lower), upper)` to clip each value.

</details>

In [ ]:
# ✏️ [EX7]


### Exercise 8: Safe Float Conversion

Given `raw = ['22.5', 'abc', '19,8', '', 'N/A', '25.0 C', None, '?', '21.0', '23,5']`, convert each to float using `safe_float()`. Print results and count how many succeeded vs failed.

**Expected output:** Should succeed for: 22.5, 19.8, 25.0, 21.0, 23.5. Rest return None.

<details><summary>💡 Hint</summary>

Call `safe_float(v)` for each and count None vs non-None results.

</details>

In [ ]:
# ✏️ [EX8]


### Exercise 9: Normalize Category Strings

Given statuses `['OK', 'ok', ' Ok ', 'O.K.', 'warning', 'ERROR', 'err', '', None, 'WARN']`, normalize each to one of `{'OK', 'WARN', 'CRITICAL'}` using a mapping dict. Print original and normalized values.

**Expected output:** OK/ok/Ok/O.K. -> OK, warning/WARN -> WARN, ERROR/err -> CRITICAL, empty/None -> None.

<details><summary>💡 Hint</summary>

Create a mapping dict and use `normalize_category()`.

</details>

In [ ]:
# ✏️ [EX9]


### Exercise 10: Min-Max Normalize

Given `scores = [45, 67, 78, 92, 55, 81, 70, 88]`, apply min-max normalization. Print each original score alongside its normalized value (0 to 1).

**Expected output:** 45 -> 0.0, 92 -> 1.0, others proportionally between.

<details><summary>💡 Hint</summary>

Use formula: `(x - min) / (max - min)`.

</details>

In [ ]:
# ✏️ [EX10]


### Exercise 11: Z-Score Standardize

Given `measurements = [100, 105, 98, 110, 97, 103, 108, 101, 99, 106]`, compute z-scores for each. Print original and z-score. Identify which are more than 1 std away from mean.

**Expected output:** Mean ~102.7, values far from mean will have |z| > 1.

<details><summary>💡 Hint</summary>

Compute mean and std first, then `z = (x - mean) / std`.

</details>

In [ ]:
# ✏️ [EX11]


### Exercise 12: Multi-Column Cleaning

Given data with columns `temp` (some missing, some 'N/A'), `humidity` (some out of range like 150%), `status` (inconsistent cases): clean all three columns. Print a before/after summary.

**Expected output:** Show count of issues found and fixed in each column.

<details><summary>💡 Hint</summary>

Apply safe_float to numeric cols, normalize_category to status, check ranges.

</details>

In [ ]:
# ✏️ [EX12]


### Exercise 13: Outlier Report

Create a dataset of 20 sensor readings with 3 injected outliers. Use BOTH IQR and z-score methods. Print a comparison: which outliers each method found and any differences.

**Expected output:** Both methods should catch extreme outliers; borderline cases may differ.

<details><summary>💡 Hint</summary>

Use `random.seed(42)` and inject known outliers, then run both detections.

</details>

In [ ]:
# ✏️ [EX13]


### Exercise 14: Quality Score Calculator

Write a function `data_quality_score(data, columns)` that returns a score from 0 to 100 based on: completeness (% non-missing), validity (% in valid range), consistency (% matching expected types). Test on a messy dataset.

**Expected output:** Something like: completeness=80%, validity=90%, consistency=75%, overall=81.7%.

<details><summary>💡 Hint</summary>

Compute each metric separately and average them.

</details>

In [ ]:
# ✏️ [EX14]


### Exercise 15: Full Cleaning Pipeline

Build a complete pipeline for this data: `[{'id':'S01','temp':'22.5','hum':'55','status':'OK'},{'id':'S02','temp':'N/A','hum':'42','status':'warn'},{'id':'S03','temp':'99.9','hum':'','status':'OK'},{'id':'S04','temp':'abc','hum':'38','status':'ERROR'},{'id':'S05','temp':'21.0','hum':'61','status':'ok'},{'id':'S06','temp':'23.1','hum':'150','status':'OK'},{'id':'S07','temp':'22.8','hum':'52','status':''},{'id':'S08','temp':'20.5','hum':'nan','status':'OK'}]`. Steps: (1) convert types, (2) fill missing humidity with mean, (3) drop rows with missing temp, (4) detect+clip temp outliers, (5) normalize temp to [0,1], (6) normalize status strings. Print the final clean dataset.

**Expected output:** Show each pipeline stage with row count and changes made.

<details><summary>💡 Hint</summary>

Chain the functions: safe_float -> fill_with_mean -> clip_outliers -> min_max_normalize -> normalize_category.

</details>

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

You now have a complete data processing toolkit: **read** structured data (CSV/JSON), **validate** it against a schema, **clean** missing values and outliers, and **normalize** scales. Next week we move to **functions as pipeline stages** — wrapping each processing step into reusable, composable functions that can be chained together into elegant, maintainable data pipelines.